In [ ]:
# Install the Gymnasium and stable_baselines3
!pip install gymnasium stable_baselines3

In [ ]:
import gymnasium as gym
import stablebaseline3 as sb3
import torch as th
import numpy as np


# We are reusing the previus MDP

# The MDP has the following properties. It illustrates the use of Reinforcement Learning to control the cooling of a big fridge at a grocery store.
#
# State Space: 2 floats, time_of_day (from 0 at midnight to 1 at midnight the next day) and current_temperature (from -20 to -5).
# Action Space: 1 float, cooling strength (from 0 to 1)
# Effect of the actions:
# - One episode is one day
# - One day is 24 hours. The agent controls every hour (so, 24 time-steps per episode)
# - At the beginning of an episode, the fridge temperature is a random number between -19 and -16 (it is a float)
# - Fridges warm up when food is put in them and the door is opened. The amount of watts (W) of heat added to the fridge, for every hour, is in WATTS_IN. This needs to be multiplied with a random number between 0.9 and 1.1
# - Refrigeration units are able to cool down from 0 watts (action = 0) to 500 watts (action = 1)
# - To have this cooling action, the refrigeration unit consumes an amount of electrical power equal to cooling [see point above] * 2.0 * exp(-fridge_temperature / 20   -   1)
# - After an hour, the temperature of the fridge will have changed by (watts in - watts cooled) * THERMAL_INERTIA
# - You now have a new temperature :)
# Reward function: The reward is equal to 1 - electrical consumption during the time-step / 1000
# Termination function: The episode terminates if the temperature of the fridge goes below -20C or above -15C. It also terminates after 24 hours.

WATTS_IN = [50, 55, 38, 41, 800, 750, 450, 300, 450, 100, 210, 600, 450, 120, 80, 350, 220, 810, 420, 250, 80, 45, 61, 55]
THERMAL_INERTIA = 0.002

class FridgeEnv(gym.Env):
  def __init__(self):
    self.observation_space = gym.spaces.Box(low=0.0, high=1.0, shape=(2,), dtype=np.float32)
    self.action_space = gym.spaces.Box(low=0.0, high=1.0, shape=(1,), dtype=np.float32)

  def reset(self, **kwargs):
    self.time_of_day = 0
    self.fridge_temperature = -19.0 + np.random.random() * 3.0

    return self.make_state(), {}

  def step(self, action):
    # Opening the door warms up the fridge
    watts_entering_fridge = WATTS_IN[self.time_of_day]
    watts_entering_fridge *= 0.9 + np.random.random() * 0.2

    # The cooling system cools down the fridge
    watts_leaving_fridge = action[0] * 500.0

    # Electrical power consumed
    watts_of_electricity = watts_leaving_fridge * 2.0 * np.exp(-self.fridge_temperature / 20.0 - 1.0)

    # Update temperature and time of day
    self.fridge_temperature += (watts_entering_fridge - watts_leaving_fridge) * THERMAL_INERTIA
    self.time_of_day = (self.time_of_day + 1) % 24

    # Compute reward
    reward = 1.0 - watts_of_electricity / 1000

    # Done and truncated
    done = (self.fridge_temperature > -15.0) or (self.fridge_temperature < -20.0)
    truncated = self.time_of_day == 0 # We went back to hour 0, so 24 hours passed

    return self.make_state(), reward, done, truncated, {}

  def make_state(self):
    return np.array([
        self.time_of_day / 24.0,
        self.fridge_temperature
    ])

# Test one episode
env = FridgeEnv()
env = gym.wrappers.TransformReward(env, ...)

done = False
state, _ = env.reset()

while not done:
  action = env.action_space.sample()
  state, reward, done, truncated, _ = env.step(action)

  print(state, action, reward)

  done = done | truncated

In [ ]:
class Actor_network(th.nn.Module):
    def __init__(obs_space, action_space):
        self.nn = th.nn.Sequential(
            th.nn.Linear(obs_dim, 4),
            th.nn.ReLU(),
            th.nn.Linear(4, 8),
            th.nn.ReLU(),
            th.nn.Linear(8, action_dim),
            th.nn.Softmax(-1))
    
    def forward(obs, action):
        x = th.cat([obs, action], dim=1)
        return self.nn(x)        

class DDPG: 
    def __init__(env,
                 learning_rate=2.5e-4
                 buffer_size=1000000,
                 learning_starts=96,
                 batch_size=64,
                 tau=0.95,
                 gamma=1,
                 train_freq=4,
                 target_update_freq=10000):
        self.learning_starts = learning_starts
        self.train_freq = train_freq
        self.target_update_freq = target_update_freq
        self.tau = tau
        
        self.q_network = Q_network(env.observation_space, env.action_space)
        self.q_network_target = Q_network(env.observation_space, env.action_space)
        self.optimizer = th.optim.Adam(self.q_network1.parameters(), lr=self.learning_rate)
        self.rb = sb3.common.buffers.ReplayBuffer(buffer_size,
                env.observation_space,
                env.action_space,
                n_envs=1,
                optimize_memory_usage=False)

    def learn(total_timesteps):        
        for i in total_timesteps:
            if random.random() < 0.2:
                # Epsilon greedy action
              
            else:
                # Q_learning action selection
            
            # Fill the replay buffer
            
            
            if global_step > self.learning_starts:
                if global_step % self.train_freq == 0:
                    data = rb.sample(args.batch_size)
                    # Implement DQN update algorithm

                # Target Network Update
                if global_step % self.target_update_freq == 0:
                    for target_network_param, q_network_param in zip(self.q_network_target.parameters(), self.q_network.parameters()):
                        target_network_param.data.copy_(
                            self.tau * q_network_param.data + (1.0 - self.tau) * target_network_param.data
                        )


In [ ]:
# Use DQN to learn in the environment for 20K time-steps
env = FridgeEnv()
env = gym.wrappers.RecordEpisodeStatistics(env)  # Allows to get nice learning curves without using Tensorboard

agent = DQN(
    env=env
)

agent.learn(total_timesteps=20_000)

%pylab inline
import matplotlib.pyplot as plt

plt.plot(env.return_queue)